# Is GFTM's A_par phase winding physics or a Hermite-basis artefact? (Fusion_PhD-dtg2)

GFTM tearing-parity modes on NSTX_MTM have exact parity (pyro's `apar_even_fraction` = 1) but a tearing parameter
T = |∫A dl| / ∫|A| dl well below GS2's, because A_par's phase winds along θ (Fusion_PhD-1yzb). Exact parity is built into
the Hermite basis, so parity cannot tell whether the winding is physical. This notebook follows T, the even fraction, the
|A|²-weighted phase spread and γ along a GFTM convergence ladder built by GyroRun (project `MTM_PHASE_WINDING`):
(NBASIS_MAX, NXGRID) ∈ {(24,30), (34,40), (44,50), (59,65)} × WIDTH ∈ {3.0, 4.5, 5.893, 8.0}, every other setting as
m11_new, on all 300 NSTX_MTM n300 cases. The (24, 30, 5.893) rung *is* m11_new.

If the winding shrinks and T rises toward GS2 with resolution or width, it is a basis artefact; if T holds as the basis
converges, GFTM's eigenfunction differs from GS2's. Advisory numbers only.

## Imports and settings

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from pyrokinetics import PyroHypercube
from general_analysis import mode_classification as mc

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "gftm_phase_winding"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
gftm_root = data_root / "GFTM/Runs/MTM_PHASE_WINDING/NSTX_MTM"
gs2_root = data_root / "GS2/Runs/LATIN_HYPERCUBE/NSTX_MTM/beta_q_shat_ky_n300"
resolutions = [(24, 30), (34, 40), (44, 50), (59, 65)]   # (NBASIS_MAX, NXGRID), NBASIS_MAX = NXGRID - 6
widths = {3.0: "3p0", 4.5: "4p5", 5.89285: "5p893", 8.0: "8p0"}
m11 = (24, 30, 5.89285)
n_overlay = 6                  # overlay cases: sample 1, then the largest GS2-GFTM gap in T at m11
must_include = "iteration_1"
tearing_cut = 0.5              # apar_even_fraction > 0.5 = tearing parity (user, 2026-09-30)
leaf = lambda nb, nx, w: gftm_root / f"nb{nb}_nx{nx}_w{widths[w]}"
settings = [(nb, nx, w) for nb, nx in resolutions for w in widths]

## Load data

Every leaf through pyrokinetics (`PyroHypercube` + GyroRun's `pyroscan.nc`, whose save job stored `tearing_parameter`
and `apar_even_fraction` per run and mode). GS2's T and even fraction come from `mc.indicators(scan, ds)` on the n300 tail-averaged cube + its
`pyroscan.json` (each sample's Pyro rebuilt in memory, `FieldLine` on its apar; no run directory is read, Fusion_PhD-use8). n300 has no final-time cube,
so gamma, T and E are the cube's tail-averaged values, not the last time point the per-run version used.

In [ ]:
cubes = {}
for s in settings:
    cube = PyroHypercube(pyroscan_json=leaf(*s) / "pyroscan.json", load_base_pyro=True)
    cube.load_gk_output(netcdf_file=leaf(*s) / "pyroscan.nc")
    cubes[s] = cube.gk_output.data
print(cubes[m11])

In [ ]:
def last(da):
    da = da.pint.dequantify() if hasattr(da, "pint") else da
    return da.isel(time=-1) if "time" in da.dims else da

scan, ds = mc.load_gs2_cube(gs2_root, "pyro_cube_avg")
gs2_ds = ds
ind = mc.indicators(scan, ds)
gs2 = {str(n): dict(gamma=float(g), E=float(e), T=float(t)) for n, g, e, t in
       zip(ind.sample_name.values, last(ds["growth_rate"]).squeeze().values, ind["E"].values, ind["T"].values)}
print(len(gs2), "GS2 cases")

## Calculate and select

Per sample and setting, the **tearing branch** is the fastest mode with even fraction > 0.5 (argmax of γ over those
modes, never a fixed mode index); `dominant` records whether it is also the fastest mode overall. The population is every
case where GS2 is unstable with tearing parity and m11_new's fastest mode is tearing parity.

In [ ]:
def plain(da):
    return da.pint.dequantify() if hasattr(da, "pint") else da

def per_sample(ds):
    """(gamma, E, T, dominant, mode index) of the tearing branch per sample."""
    g, E, T = (plain(ds[v]) for v in ("growth_rate", "apar_even_fraction", "tearing_parameter"))
    tear_g = g.where(E > tearing_cut)
    ok = tear_g.notnull().any("mode")
    idx = tear_g.fillna(-np.inf).argmax("mode")
    pick = lambda x: x.isel(mode=idx).where(ok)
    dominant = (idx == g.fillna(-np.inf).argmax("mode")) & ok
    return pick(g), pick(E), pick(T), dominant, idx

branch = {s: per_sample(cubes[s]) for s in settings}
names = [str(n) for n in cubes[m11]["sample_name"].values]
assert all([str(n) for n in cubes[s]["sample_name"].values] == names for s in settings)

In [ ]:
g11, E11, T11, dom11, _ = branch[m11]
population = [n for i, n in enumerate(names)
              if n in gs2 and gs2[n]["gamma"] > 0 and gs2[n]["E"] > tearing_cut and bool(dom11[i])]
gap = {n: gs2[n]["T"] - float(T11[names.index(n)]) for n in population}
overlay = [must_include] + [n for n in sorted(gap, key=gap.get, reverse=True) if n != must_include][: n_overlay - 1]
print(f"population: {len(population)} of {len(names)} cases (GS2 unstable + tearing parity, m11_new fastest mode tearing)")
for n in overlay:
    print(f"{n}: GS2 gamma={gs2[n]['gamma']:.4f} T={gs2[n]['T']:.3f} | m11_new gamma={float(g11[names.index(n)]):.4f} "
          f"T={float(T11[names.index(n)]):.3f}")

### Eigenfunctions of the overlay cases

GFTM's θ-resolved A_par comes from each leaf's `pyroscan.nc`; GS2's is read run by run. Phase is unwrapped along θ and referenced to θ = 0; the spread is the |A|²·dθ-weighted standard deviation of that phase, over all θ
and over the core |θ| < π only. `core_weight` is the fraction of ∫|A|²dθ inside |θ| < π.

In [ ]:
def gs2_apar(n):
    a = last(plain(gs2_ds["apar"])).isel(sample=list(gs2_ds.sample_name.values.astype(str)).index(n)).squeeze()
    return a["theta"].values, a.values

def gftm_apar(s, i):
    a = plain(cubes[s]["apar"]).isel(sample=i, kx=0, mode=int(branch[s][4][i]))
    return a["theta"].values, a.values

def phase_metrics(theta, a):
    w = np.abs(a) ** 2 * np.gradient(theta)
    ph = np.unwrap(np.angle(a))
    ph = ph - np.interp(0.0, theta, ph)
    spread = lambda m: np.sqrt(np.average((ph[m] - np.average(ph[m], weights=w[m])) ** 2, weights=w[m]))
    core = np.abs(theta) < np.pi
    return dict(phase=ph, spread=spread(np.ones_like(core)), spread_core=spread(core), core_weight=w[core].sum() / w.sum())

fields = {}
for n in overlay:
    i = names.index(n)
    fields[n, "GS2"] = gs2_apar(n)
    for s in settings:
        if np.isfinite(float(branch[s][0][i])):
            fields[n, s] = gftm_apar(s, i)
metrics = {k: phase_metrics(*v) for k, v in fields.items()}

print(f"{'case':14s} {'setting':18s} {'gamma':>8s} {'E':>6s} {'T':>6s} {'spread':>7s} {'core':>6s} {'core_wt':>7s} dom")
for n in overlay:
    m = metrics[n, "GS2"]
    print(f"{n:14s} {'GS2':18s} {gs2[n]['gamma']:8.4f} {gs2[n]['E']:6.3f} {gs2[n]['T']:6.3f} {m['spread']:7.2f} {m['spread_core']:6.2f} {m['core_weight']:7.3f}")
    for s in settings:
        if (n, s) not in metrics:
            continue
        i = names.index(n); g, E, T, dom, _ = (x[i] for x in branch[s]); m = metrics[n, s]
        print(f"{'':14s} nb{s[0]} nx{s[1]} W={s[2]:<7g} {float(g):8.4f} {float(E):6.3f} {float(T):6.3f} {m['spread']:7.2f} "
              f"{m['spread_core']:6.2f} {m['core_weight']:7.3f} {bool(dom)}")

## Plot

In [ ]:
res_colours = plt.cm.viridis(np.linspace(0, 0.85, len(resolutions)))
w_colours = plt.cm.plasma(np.linspace(0, 0.85, len(widths)))
top = resolutions[-1]
figs = {}
for n in overlay:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    for col, (title, series) in enumerate([
        (f"resolution ladder at W={m11[2]:g}", [((nb, nx, m11[2]), f"nb{nb}/nx{nx}", c) for (nb, nx), c in zip(resolutions, res_colours)]),
        (f"width scan at nb{top[0]}/nx{top[1]}", [((*top, w), f"W={w:g}", c) for w, c in zip(widths, w_colours)]),
    ]):
        th, a = fields[n, "GS2"]
        axes[0, col].plot(th / np.pi, np.abs(a) / np.abs(a).max(), "k", lw=2, label="GS2")
        axes[1, col].plot(th / np.pi, metrics[n, "GS2"]["phase"], "k", lw=2)
        for s, label, c in series:
            if (n, s) not in fields:
                continue
            th, a = fields[n, s]
            axes[0, col].plot(th / np.pi, np.abs(a) / np.abs(a).max(), color=c, label=label)
            axes[1, col].plot(th / np.pi, metrics[n, s]["phase"], color=c)
        axes[0, col].set(title=title, ylabel="|A_par| / max")
        axes[1, col].set(xlabel="θ / π", ylabel="arg A_par − arg A_par(0)  [rad]")
        axes[0, col].legend(fontsize=8)
        for ax in axes[:, col]:
            ax.axvspan(-1, 1, color="0.9", zorder=0)
    fig.suptitle(f"NSTX_MTM n300 {n}: A_par, GS2 vs GFTM tearing branch (shaded |θ|<π)")
    figs[f"eigenfunction_{n}"] = fig

In [ ]:
nbs = [nb for nb, _ in resolutions]
rows = [("T", "tearing parameter T"), ("spread", "phase spread [rad]"), ("E", "apar even fraction"), ("gamma", "γ / γ(m11_new)")]
fig, axes = plt.subplots(len(rows), len(overlay), figsize=(3.2 * len(overlay), 10), sharex=True, squeeze=False)
for j, n in enumerate(overlay):
    i = names.index(n)
    for (w, c) in zip(widths, w_colours):
        vals = {"T": [], "spread": [], "E": [], "gamma": []}
        for nb, nx in resolutions:
            g, E, T = (float(x[i]) for x in branch[nb, nx, w][:3])
            vals["T"].append(T); vals["E"].append(E); vals["gamma"].append(g / float(g11[i]))
            vals["spread"].append(metrics[n, (nb, nx, w)]["spread"] if (n, (nb, nx, w)) in metrics else np.nan)
        for r, (k, _) in enumerate(rows):
            axes[r, j].plot(nbs, vals[k], "o-", color=c, label=f"W={w:g}")
    for r, (k, lab) in enumerate(rows):
        ref = {"T": gs2[n]["T"], "E": gs2[n]["E"], "spread": metrics[n, "GS2"]["spread"], "gamma": gs2[n]["gamma"] / float(g11[i])}[k]
        axes[r, j].axhline(ref, color="k", ls="--", label="GS2")
        axes[r, 0].set_ylabel(lab)
    axes[0, j].set_title(n)
    axes[-1, j].set_xlabel("NBASIS_MAX (NXGRID = +6)")
axes[0, 0].legend(fontsize=7)
fig.suptitle("Tearing branch along the ladder (dashed: GS2)")
figs["ladder_overlay_cases"] = fig

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
gs2_T = np.array([gs2[n]["T"] for n in population])
idx = [names.index(n) for n in population]
for w, c in zip(widths, w_colours):
    med, lo, hi, gr, keep = [], [], [], [], []
    for nb, nx in resolutions:
        g, E, T, dom, _ = branch[nb, nx, w]
        t = np.array([float(T[i]) for i in idx])
        med.append(np.nanmedian(t)); lo.append(np.nanpercentile(t, 25)); hi.append(np.nanpercentile(t, 75))
        gr.append(np.nanmedian([float(g[i]) / float(g11[i]) for i in idx]))
        keep.append(np.mean([bool(dom[i]) for i in idx]))
    axes[0].plot(nbs, med, "o-", color=c, label=f"W={w:g}")
    axes[0].fill_between(nbs, lo, hi, color=c, alpha=0.15)
    axes[1].plot(nbs, gr, "o-", color=c)
    axes[2].plot(nbs, keep, "o-", color=c)
axes[0].axhline(np.median(gs2_T), color="k", ls="--", label="GS2 median")
axes[0].set(ylabel="tearing parameter T (median, IQR)")
axes[1].set(ylabel="median γ / γ(m11_new)")
axes[2].set(ylabel="fraction with tearing branch dominant")
for ax in axes:
    ax.set_xlabel("NBASIS_MAX (NXGRID = +6)")
axes[0].legend(fontsize=8)
fig.suptitle(f"Population: {len(population)} NSTX_MTM n300 cases, GS2 tearing + m11_new fastest mode tearing")
figs["ladder_population"] = fig
plt.show()

## Save

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
for name, fig in figs.items():
    fig.savefig(output_dir / f"{name}.png", dpi=150, bbox_inches="tight")
print(sorted(p.name for p in output_dir.glob("*.png")))

## Interpretation

Filled in after the run, from the printed tables above: see Fusion_PhD `results/gftm_phase_winding/README.md`.